In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
import seaborn as sns
import matplotlib as mpl
mpl.rcParams['legend.markerscale'] = 1.5
mpl.rcParams['pdf.fonttype'] = 42
import matplotlib.pyplot as plt
import numpy as np
from utils.optimal_transport import plot_transition_matrix
from utils.mypalette import condition_palette, mutation_palette, cluster_palette_dict, day_palette, palette_mutcond
from utils.sankey_flow import plot_sankey_flow
import ot
from sklearnex import patch_sklearn
patch_sklearn()

warnings.filterwarnings('ignore')

x = '240813_5'
single_exps = ['E1']

saving_dir = f'../results/{x}/clustering/{("_".join(single_exps))}/CTRL_VCP_partial_OT'
saving_dir = os.path.join(saving_dir, ("_".join(single_exps)))
os.makedirs(saving_dir, exist_ok=True)

data_root = f"../embeddings/{x}/"
emb_path = data_root + f"{('_'.join(single_exps))}_emb.feather"

annotations_df = pd.read_feather(emb_path)
emb_cols = [col for col in annotations_df.columns if col.startswith('emb')]

raw_emb = annotations_df[emb_cols].values
annotations_df['condition'] = annotations_df['condition'].str.replace("A1 ", "TIC")
annotations_df['mut_cond'] = annotations_df['mut_cond'].str.replace("A1 ", "TIC")
annotations_df['cluster'] = annotations_df['cluster'].astype('category')
unique_clusters = sorted(annotations_df['cluster'].unique())

# Process data to analyze relationship between VCP untreated and all CTRL
annotations_df['cluster_mutcond'] = annotations_df['mut_cond'] + '_' + annotations_df['cluster'].astype(str)
annotations_df['cluster_mutcond'] = annotations_df['cluster_mutcond'].astype('category')
print(annotations_df['cluster_mutcond'].nunique())

cluster_palette = cluster_palette_dict(unique_clusters)

### Useful functions

In [ ]:
def get_group_mass(labels):
    unique, counts = np.unique(labels, return_counts=True)
    group_weights = {g: 1.0 / len(unique) / c for g, c in zip(unique, counts)}
    return np.array([group_weights[l] for l in labels])


def compute_state_transition_matrix(
    dataframe,
    points,
    category_column,
    celltype_column=None,
    category_1=None,
    category_2=None,
    cost_metric='euclidean',
    seed=42,
    correct_group_sizes=False,
    ot_type='partial',
    reg_m=None,
    reg=0.,
    m=None,
    celltype_ratio_threshold=0.05,
):
    """
    Computes an OT-based transition matrix between clusters or cell types.

    - If `category_1`, `category_2`, and `celltype_column` are all provided:
        Computes OT between individual points in the two categories, aggregated by cell type.
    - If any of these are None:
        Performs a stratified random split of the dataset by `category_column`,
        and computes point-wise OT between the two halves, aggregated by category.

    Parameters:
    - ot_type: Type of OT to use. Options are 'partial', 'unbalanced', or 'exact'.
    - reg_m: Marginal relaxation parameter for unbalanced OT.
    - reg: Entropic regularization parameter for unbalanced OT.
    """

    rng = np.random.default_rng(seed)

    ot_type = str(ot_type).lower()
    assert ot_type in ['partial', 'unbalanced', 'exact'], f"Invalid ot_type: {ot_type}"
    if ot_type == 'partial':
        assert m is not None, "For partial OT, m (partial matching mass) must be specified."
    elif ot_type == 'unbalanced':
        assert reg_m is not None, "For unbalanced OT, reg_m (marginal relaxation parameter) must be specified."

    use_random_split = (category_1 is None or category_2 is None or celltype_column is None)

#    if use_random_split:
#        # Random split stratified by category_column
#        dataframe = dataframe.copy()
#        dataframe['split'] = None
#        indices_1, indices_2 = [], []
#
#        for cat in dataframe[category_column].unique():
#            idx = dataframe[dataframe[category_column] == cat].index
#            idx = rng.permutation(idx)
#            if len(idx) % 2 != 0:
#                idx = idx[:-1]
#            n_half = len(idx) // 2
#            indices_1.extend(idx[:n_half])
#            indices_2.extend(idx[n_half:])
#
#        dataframe.loc[indices_1, 'split'] = 'part_1'
#        dataframe.loc[indices_2, 'split'] = 'part_2'
#
#        df1 = dataframe[dataframe['split'] == 'part_1']
#        df2 = dataframe[dataframe['split'] == 'part_2']
#        # Verify that the two splits have the same distributions of category_column labels
#        split_1_dist = df1[category_column].value_counts(normalize=True)
#        split_2_dist = df2[category_column].value_counts(normalize=True)
#
#        if not np.allclose(split_1_dist, split_2_dist, rtol=1e-5):
#            raise ValueError("The two splits do not have the same distributions of category_column labels.")
#        pts1 = points[df1.index]
#        pts2 = points[df2.index]
#
#        assert len(pts1) == len(pts2), "The two splits do not have the same number of points."
#
#        if correct_group_sizes:
#            a = get_group_mass(df1[category_column if use_random_split else celltype_column].values)
#            b = get_group_mass(df2[category_column if use_random_split else celltype_column].values)
#        else:
#            a = ot.unif(len(pts1))
#            b = ot.unif(len(pts2))
#
#        C_fwd = ot.dist(pts1, pts2, metric=cost_metric)
#        C_bwd = ot.dist(pts2, pts1, metric=cost_metric)
#
#        if ot_type == 'unbalanced':
#            P_fwd = ot.unbalanced.mm_unbalanced(a, b, C_fwd, reg_m=reg_m, reg=reg, div='kl')
#            P = P_fwd
#        elif ot_type == 'exact':
#            P_fwd = ot.emd(a, b, C_fwd)
#            P_bwd = ot.emd(b, a, C_bwd)
#            P = (P_fwd + P_bwd.T) / 2
#        #elif ot_type == 'partial':
#        #    P = ot.partial.partial_wasserstein(a, b, C_fwd, )
#
#        classes_1 = np.sort(df1[category_column].unique())
#        classes_2 = np.sort(df2[category_column].unique())
#        idx1 = {c: np.where(df1[category_column].values == c)[0] for c in classes_1}
#        idx2 = {c: np.where(df2[category_column].values == c)[0] for c in classes_2}
#
#        T = np.zeros((len(classes_1), len(classes_2)))
#        for i, c1 in enumerate(classes_1):
#            for j, c2 in enumerate(classes_2):
#                T[i, j] = P[np.ix_(idx1[c1], idx2[c2])].sum()
#
#        T = pd.DataFrame(T, index=classes_1, columns=classes_2)
#        T_non_norm = T.copy()
#        T = T.div(T.sum(axis=1), axis=0)   # Normalization --> rows are probabilities, but we lose symmetry
#
#        return {
#            'transition_matrix': T,
#            'transition_non_normalized': T_non_norm,
#            'classes_1': classes_1,
#            'classes_2': classes_2,
#            'transport_plan': P,
#            'category_column': category_column,
#            'source': 'part_1',
#            'target': 'part_2',
#            'source_emb': pts1,
#            'target_emb': pts2,
#            'annotations_df': dataframe,
#            'celltype_column': category_column,
#        }

    #else:
    # compare category_1 vs category_2 and aggregate by cell type
    df1 = dataframe[dataframe[category_column] == category_1]
    df2 = dataframe[dataframe[category_column] == category_2]

    if celltype_column is not None:
        ratio1 = df1[celltype_column].value_counts(normalize=True)
        ratio2 = df2[celltype_column].value_counts(normalize=True)
        valid_types1 = ratio1[ratio1 >= celltype_ratio_threshold].index
        valid_types2 = ratio2[ratio2 >= celltype_ratio_threshold].index
        #valid_types = set(valid_types1).intersection(valid_types2)
        df1 = df1[df1[celltype_column].isin(valid_types1)]
        df2 = df2[df2[celltype_column].isin(valid_types2)]
        
    pts1 = points[df1.index]
    pts2 = points[df2.index]

    if ot_type != 'partial':
        if correct_group_sizes:
            a = get_group_mass(df1[category_column if use_random_split else celltype_column].values)
            b = get_group_mass(df2[category_column if use_random_split else celltype_column].values)
        else:
            a = ot.unif(len(pts1))
            b = ot.unif(len(pts2))

    elif ot_type == 'partial':
        a = np.ones(len(pts1))
        b = np.ones(len(pts2))
        min_m = min(len(pts1), len(pts2))
        if isinstance(m, str) and m.lower() in ['min', 'minimum']:
            calc_m = min_m
        # If m is a fraction like 0.90 (transport 90% of the minimum between the two sets)
        elif isinstance(m, (float, int)) and 0 < m <= 1.0:
            calc_m = int(m * min_m)

    C = ot.dist(pts1, pts2, metric=cost_metric)

    if ot_type == 'unbalanced':
        P = ot.unbalanced.mm_unbalanced(a, b, C, reg_m=reg_m, reg=reg, div='kl')
    elif ot_type == 'exact':
        P = ot.emd(a, b, C)
    elif ot_type == 'partial':
        P = ot.partial.partial_wasserstein(a, b, C, m=calc_m)

    classes_1 = np.sort(df1[celltype_column].unique())
    classes_2 = np.sort(df2[celltype_column].unique())
    idx1 = {c: np.where(df1[celltype_column].values == c)[0] for c in classes_1}
    idx2 = {c: np.where(df2[celltype_column].values == c)[0] for c in classes_2}

    T = np.zeros((len(classes_1), len(classes_2)))
    for i, c1 in enumerate(classes_1):
        for j, c2 in enumerate(classes_2):
            T[i, j] = P[np.ix_(idx1[c1], idx2[c2])].sum()

    T = pd.DataFrame(T, index=classes_1, columns=classes_2)
    T = T.div(T.sum(axis=0), axis=1)

    return {
        'transition_matrix': T,
        'classes_1': classes_1,
        'classes_2': classes_2,
        'transport_plan': P,
        'cost_matrix': C,
        'category_column': category_column,
        'source': category_1,
        'target': category_2,
        'source_emb': pts1,
        'target_emb': pts2,
        'annotations_df': dataframe,
        'celltype_column': celltype_column,
    }


## State transition matrix and 'flow' visualization

### Flows from CTRL (untr) to VCP (untr)

In [ ]:
cost_matrices = {}

treatment_flows = pd.DataFrame(columns=['transition_matrix',
            'classes_1',
            'classes_2',
            'transport_plan',
            'cost_matrix',
            'category_column',
            'source',
            'target',
            'celltype_column',
            'annotations_df',
        ])

for condition in ['CTRL_untreated', 'CTRL_TIClow', 'CTRL_TIChigh']:
    if not (condition in annotations_df['mut_cond'].unique()):
        continue
    # 1. Compute the base transition matrix
    temp_res = compute_state_transition_matrix(
        annotations_df,
        raw_emb,
        'mut_cond',
        'cluster',
        condition,
        'VCP_untreated',
        cost_metric='euclidean',
        correct_group_sizes=False,
        ot_type='partial',
        #reg_m = 1,
        m=1.0,
    )

    cost_matrices[condition] = temp_res['cost_matrix']

#fig, ax = plt.subplots(1, 3, figsize=(15, 5))
#for i, (condition, cost_matrix) in enumerate(cost_matrices.items()):
#    sns.histplot(cost_matrix.flatten(), bins=100, ax=ax[i], label=condition)
#    ax[i].set_title(f'Costs - {condition}\nmedian = {np.median(cost_matrix.flatten()):.2f}')
#    ax[i].set_xlabel('Cost')
#    ax[i].set_ylabel('Frequency')

## sankey plots for CTRL conditions --> VCP untreated

In [ ]:
#edge_threshold = 0.0

for condition in ['CTRL_untreated', 'CTRL_TIClow', 'CTRL_TIChigh']:
    
    if not (condition in annotations_df['mut_cond'].unique()):
        continue

    results = compute_state_transition_matrix(
        annotations_df,
        raw_emb,
        'mut_cond',
        'cluster',
        condition,
        'VCP_untreated',
        cost_metric='euclidean',
        correct_group_sizes=False,
        ot_type='partial',
        m=1.0,
    )

    # Store results in your dataframe
    treatment_flows.loc[condition] = results
    
    # 2. Extract and Normalize
    # We work on a copy to avoid overwriting the original raw data
    tm = results['transition_matrix'].copy()

    #fig, ax = plt.subplots(figsize=(8, 8))
    #sns.histplot(tm.values.flatten(), bins=100, kde=True)
    #plt.title(f"Distribution of Transition Probabilities ({condition})")
    #plt.xlabel("Transition Probability")
    #plt.ylabel("Frequency")
    #plt.show()

    print(f"Mean of Transition Probabilities for {condition}: {tm.values.flatten().mean()}")
    edge_threshold = 0.5 * tm.values.flatten().mean()
    print(f"Edge threshold for {condition}: {edge_threshold}")

    # THRESHOLDING
    # Thresholding: Remove noise before normalizing
    tm[tm < edge_threshold] = 0
    
    # Twice Normalization Logic:
    # First, normalize by rows (Source) so total flow is 1
    tm = tm.div(tm.sum(axis=1), axis=0).fillna(0)
    
    # Second, normalize by columns (Target) to get "Uniform Bar Heights" 
    # This makes every target cluster bar the same size in the Sankey
    twice_normalized_tm = tm.div(tm.sum(axis=0), axis=1).fillna(0)
    
    print(f"--- Stats for {condition} ---")
    print("Col sums (Should be 1.0 for uniform bars):\n", twice_normalized_tm.sum(axis=0))

    # 3. Create a display object for the Sankey
    # We copy the results dict so we don't break the original 'results' object
    display_data = results.copy()
    display_data['transition_matrix'] = twice_normalized_tm
    
    # 4. Plot the Normalized Sankey
    plot_sankey_flow(
        display_data,
        cluster_palette,
        size_threshold=0.0,
        edge_threshold=0.0, # keep 0.0 because you already normalized after edge removal
        title=f"Sankey: Normalized Target Flows ({condition})",
        save_name=f"sankey_VCP_{condition}_CTRL.svg",
        saving_dir = saving_dir
    )

    # for each target cluster, make a stacked barplot representing incoming flows
    df_stacked = twice_normalized_tm.T 

    # Plotting
    ax = df_stacked.plot(
        kind='bar', 
        stacked=True, 
        figsize=(8, 8), 
        color=[cluster_palette.get(x, '#333333') for x in df_stacked.columns],
        #edgecolor='white',
        width=0.8
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.title(f"Composition of Incoming Flow per Target Cluster ({condition})")
    plt.xlabel("Target Cluster")
    plt.ylabel("Relative Flow Contribution (Normalized)")
    plt.xticks(rotation=90)
    plt.legend(title="Source Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(saving_dir, f"stacked_barplot_VCP_{condition}_CTRL.pdf"), dpi=300, bbox_inches='tight')
    plt.show()

## Study relationship between CTRL (all) and VCP untreated

In [ ]:
VCPuntr_df = annotations_df[~annotations_df['mut_cond'].isin(['VCP_TIChigh', 'VCP_TIClow'])]
str_cluster_palette = {str(k): v for k, v in cluster_palette.items()}

VCPuntr_df['cond_cluster'] = [row['condition'] if row['als_label']=='control' else row['cluster'] for _, row in VCPuntr_df.iterrows()]

In [ ]:
atlas_simple = compute_state_transition_matrix(
    dataframe=VCPuntr_df,
    points=raw_emb,
    category_column='als_label',
    celltype_column="cond_cluster",
    category_1="control",
    category_2="als",
    cost_metric='euclidean',
    correct_group_sizes=False,
    ot_type = 'partial',
    m = 1.0,
)

fig, ax = plot_transition_matrix(
    atlas_simple,
    cluster_palette,
    title=f'Transition Matrix - {atlas_simple["source"]} to {atlas_simple["target"]}',
    figsize=(12, 8),
    #saving_dir = saving_dir,
    #saving_name = "sankey_VCP_CTRL_all.svg"
)



In [ ]:
cluster_palette['TIChigh'] = condition_palette['TIChigh']
cluster_palette['TIClow'] = condition_palette['TIClow']
cluster_palette['untreated'] = condition_palette['untreated']

tm = atlas_simple['transition_matrix'].copy()

print(f"Mean of Transition Probabilities: {tm.values.flatten().mean()}")
edge_threshold = 0.5 * tm.values.flatten().mean()
print(f"Edge threshold: {edge_threshold}")

# THRESHOLDING
# Thresholding: Remove noise before normalizing
tm[tm < edge_threshold] = 0
    
# Twice Normalization Logic:
# First, normalize by rows (Source) so total flow is 1
tm = tm.div(tm.sum(axis=1), axis=0).fillna(0)
    
# Second, normalize by columns (Target) to get "Uniform Bar Heights" 
# This makes every target cluster bar the same size in the Sankey
twice_normalized_tm = tm.div(tm.sum(axis=0), axis=1).fillna(0)
    
print(f"--- Stats ---")
print("Col sums (Should be 1.0 for uniform bars):\n", twice_normalized_tm.sum(axis=0))

# 3. Create a display object for the Sankey
# We copy the results dict so we don't break the original 'results' object
display_data = atlas_simple.copy()
display_data['transition_matrix'] = twice_normalized_tm


plot_sankey_flow(
    atlas_simple,
    cluster_palette,
    size_threshold=0.0,
    edge_threshold=0.1,
    title=f"Sankey: {atlas_simple['source']} to {atlas_simple['target']}",
    saving_dir = saving_dir,
    save_name=f"sankey_flow_{atlas_simple['source']}_to_{atlas_simple['target']}.svg",
    color_edge_according_to='source'
)

# for each target cluster, make a stacked barplot representing incoming flows
df_stacked = twice_normalized_tm.T 

# Plotting
ax = df_stacked.plot(
    kind='bar', 
    stacked=True, 
    figsize=(8, 8), 
    color=[cluster_palette.get(x, '#333333') for x in df_stacked.columns],
    #edgecolor='white',
    width=0.8
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.title(f"Composition of Incoming Flow per Target Cluster ({condition})")
plt.xlabel("Target Cluster")
plt.ylabel("Relative Flow Contribution (Normalized)")
plt.xticks(rotation=90)
plt.legend(title="Source Cluster", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(saving_dir, "stacked_barplot_control_to_als.pdf"), dpi=300, bbox_inches='tight')
plt.show()